# Export to Ollama (GGUF)

Turns your `merged/` model into a **GGUF** file (one file: weights + tokenizer + config) and registers it in Ollama, so it runs with `ollama run` on CPU or GPU without Python.

**Steps:** 1. config → 2. get llama.cpp's converter → 3. convert to GGUF → 4. write the Modelfile → 5. `ollama create` → 6. test → 7. accuracy check.

**Needs:** Ollama installed and running, and `git`.

## 1. Config
- **`QUANT = "q8_0"`**: 8-bit, ~530 MB, almost no accuracy loss. (`"bf16"` = full precision, ~950 MB.)
- **`OLLAMA_NAME`**: the name you'll use in `ollama run`.

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS = REPO_ROOT / "notebooks"

RUN_NAME = "sft_qwen2.5-0.5b"            # change to "sft_qwen2.5-0.5b_opt" after the optimized run
MERGED_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "merged"
RESULTS_DIR = NOTEBOOKS / "data" / "eval" / "results" / RUN_NAME

QUANT = "q8_0"
OLLAMA_NAME = "qwen2.5-0.5b-instruct-fc"

GGUF_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "gguf"           # gitignored
GGUF_FILE = GGUF_DIR / f"{OLLAMA_NAME}-{QUANT}.gguf"
LLAMA_CPP = REPO_ROOT / "third_party" / "llama.cpp"             # gitignored
GGUF_DIR.mkdir(parents=True, exist_ok=True)
print("Model folder:", MERGED_DIR, "| exists:", MERGED_DIR.exists())

## 2. Get llama.cpp's converter
We only need its Python script `convert_hf_to_gguf.py`; no compiling. It uses the `gguf` library bundled in the same repo, and `torch`/`transformers`, which your env already has.

⚠️ Don't `pip install -r` llama.cpp's requirements file: it can replace your CUDA PyTorch with a CPU-only build. If the script complains about one missing package, install just that one.

In [ ]:
import subprocess

if not LLAMA_CPP.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP)], check=True)
print("Converter:", (LLAMA_CPP / "convert_hf_to_gguf.py").exists())

## 3. Convert to GGUF
Reads the safetensors + config + tokenizer and writes one `.gguf` file. With `--outtype q8_0` it converts and quantizes in one go. Takes about a minute.

In [ ]:
subprocess.run([sys.executable, str(LLAMA_CPP / "convert_hf_to_gguf.py"), str(MERGED_DIR),
                "--outfile", str(GGUF_FILE), "--outtype", QUANT], check=True)
print(f"\n✅ {GGUF_FILE.name}: {GGUF_FILE.stat().st_size / 1024**2:.0f} MB "
      f"(vs {(MERGED_DIR / 'model.safetensors').stat().st_size / 1024**2:.0f} MB bf16)")

## 4. Write the Modelfile
A **Modelfile** is Ollama's recipe:
- **`FROM`**: the GGUF file.
- **`TEMPLATE`**: Qwen's chat format (`<|im_start|>role ... <|im_end|>`), same as training. Like Qwen's own template, it adds
  Qwen's default system line when the caller sends no system message.
- **No `SYSTEM` prompt baked in** (on purpose, same as the Hugging Face upload). The *caller* sends the tool-calling system prompt
  (`build_messages` does it, and so does the gateway). Baking it in forces every chat into tool mode: plain "hi" in a chat app
  then comes back as an invented call like `hello(name="world")`.
- **`PARAMETER`**: `temperature 0` (greedy, like the eval), **`repeat_penalty 1.0`** (Ollama's default 1.1 punishes the
  `"`, `{`, `:` that JSON repeats constantly), context 4096, stop at `<|im_end|>`.


In [ ]:
TEMPLATE = """{{- if not (and .Messages (eq (index .Messages 0).Role "system")) }}<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
{{ end }}{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
"""

modelfile = f'''FROM ./{GGUF_FILE.name}

TEMPLATE """{TEMPLATE}"""

PARAMETER temperature 0
PARAMETER repeat_penalty 1.0
PARAMETER num_ctx 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
'''
(GGUF_DIR / "Modelfile").write_text(modelfile, encoding="utf-8")
print(modelfile)


## 5. Register it in Ollama
`ollama create` copies the GGUF into Ollama's model store under `OLLAMA_NAME`. After this it shows up in `ollama list` and in the Ollama app's model picker.
If an old version exists, `ollama rm <name>` first, or just re-run `create`: it overwrites a model with the same name.


In [ ]:
subprocess.run(["ollama", "create", OLLAMA_NAME, "-f", "Modelfile"], cwd=GGUF_DIR, check=True)
subprocess.run(["ollama", "list"], check=True)

## 6. Test it: tool request vs plain chat
- **Tool request:** send the full trained prompt with `build_messages` (system prompt + tools + request). Expect JSON.
- **Plain chat:** "What is an LLM?" with **no** system prompt. Expect a normal text answer. This only works because no tool prompt is baked into the Modelfile.


In [ ]:
import urllib.request

from src.prompts import build_messages


def ollama_chat(messages):
    body = json.dumps({"model": OLLAMA_NAME, "messages": messages, "stream": False,
                       "options": {"temperature": 0, "repeat_penalty": 1.0}}).encode()
    req = urllib.request.Request("http://localhost:11434/api/chat", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read())["message"]["content"]


record = {
    "query": "What's the weather like in Pune right now?",
    "tools": [{"name": "get_weather", "description": "Current weather for a city.",
               "parameters": {"city": {"type": "str", "description": "City name"}}}],
}
print("WITH TOOLS :", ollama_chat(build_messages(record)))
print("PLAIN CHAT :", ollama_chat([{"role": "user", "content": "What is an LLM? One sentence."}])[:300])

## 7. Accuracy check: did GGUF/Q8 cost anything?
Run 100 questions from the frozen test set through Ollama, score them with **the same scorer**, and compare with the bf16 model's scores on the **same 100 questions**. A drop of ~1 point or less is normal for Q8_0.

In [ ]:
import pandas as pd

from src.data_utils import read_jsonl
from src.evaluate import score_prediction

N = 100
test = read_jsonl(NOTEBOOKS / "data" / "eval" / "test_v1.jsonl")[:N]
gguf_em = [score_prediction(ollama_chat(build_messages(r)), r)["exact_match"] for r in test]

bf16 = pd.read_csv(RESULTS_DIR / "scored.csv").set_index("id").loc[[r["id"] for r in test], "exact_match"]
print(f"Exact match on {N} questions:  bf16 {100 * bf16.mean():.1f}%  |  GGUF {QUANT} {100 * sum(gguf_em) / N:.1f}%")
print(f"Run it yourself:  ollama run {OLLAMA_NAME}")